# 16.01 · Capstone project — Milestone M7

**Chapter:** Capstone · **Milestone:** M7

## Goals
- [ ] Ship a real product: RAG or agent, **evals in CI**, tracing, guardrails, deployed, documented
- [ ] Work eval-first: spec → eval set → baseline → iterate with error analysis → harden → deploy
- [ ] Write a public write-up of design decisions and eval results

## Resources
- [Chip Huyen – AI Engineering (book)](https://www.oreilly.com/library/view/ai-engineering/9781098166298/) — paid
- [Hamel Husain – Your AI product needs evals](https://hamel.dev/blog/posts/evals/) · [A field guide to rapidly improving AI products](https://hamel.dev/blog/posts/field-guide/) — free
- [Eugene Yan – Patterns for building LLM-based systems](https://eugeneyan.com/writing/llm-patterns/) — free
- [Applied LLMs – What we learned from a year of building with LLMs](https://applied-llms.org/) — free

This is the course's final milestone. Everything you need was built in chapters 05–15; the capstone is about **choosing well and finishing**.

## 1. Pick a problem you actually have
Good capstones have: a real user (you count), a task where "good" can be checked, data you can get, and a scope you can ship in 4–6 weeks.

| Idea | Core chapters | How you'd evaluate it |
|---|---|---|
| Receipt / invoice → expense tracker (photos, Spanish, soles) | 14.01, 05.04, 12 | field-level accuracy on 100 labelled receipts |
| Q&A over a domain you know (regulations, product docs, course notes) with citations | 07–08, 10.04 | retrieval recall, faithfulness, answer correctness on 80 questions |
| Meeting / voice-note → action items in your task tool | 14.02, 09 | action-item precision/recall vs hand-written lists |
| Support triage bot for a small business (route, draft reply, escalate) | 09, 15.03, 11.04 | routing accuracy, reply quality (rubric judge), escalation rate |
| Coding agent for one repetitive dev task in your repos | 15.05, 09.08 | task success on 30 real past tasks, cost per task |
| Fine-tuned small model that replaces an API call in an existing app | 12, 13.01 | task score vs the API model, latency, cost |

Avoid: generic chatbots with no evaluable task, problems that need data you can't get, and "agent that does everything".

## 2. The process (and where each chapter fits)
1. **Spec (1–2 days)** — one page: user, job-to-be-done, inputs/outputs, what "good" means, constraints (latency, cost, privacy, languages). `SPEC.md`.
2. **Eval set first (3–5 days)** — 50–100 real examples with expected outputs or rubrics; split dev/test (10.02). No eval, no iteration.
3. **Baseline (1 day)** — the simplest thing: one prompt, one API model. Score it. This is the number to beat.
4. **Iterate (2–3 weeks)** — error analysis loop: run evals → read failures → bucket them → fix the biggest bucket (prompt, retrieval,
   tools, data, model) → re-run. Log every change in `DECISIONS.md` with its eval delta.
5. **Harden (1 week)** — guardrails (11.04), tracing (11.01), budgets/rate limits (13.03, 11.02), fallbacks, cost/latency measurement (13.06).
6. **Deploy (2–3 days)** — container + public URL (13.05), evals in CI, smoke tests.
7. **Write-up (2–3 days)** — what you built, why, numbers, failures, what's next (§5).

In [ ]:
import json
import os
import subprocess
import sys
import textwrap
from pathlib import Path

from aieng import DATA

## 3. Scaffold the repo
`scaffold()` writes a minimal, working project skeleton: spec/decision/write-up templates, an eval runner that **fails CI below a threshold**,
a CI workflow, and a tiny OpenAI-compatible core you replace with your system. It's written to `data/capstone/<name>` (git-ignored) — copy
it to its own repo (`cp -r data/capstone/<name> ~/Developer/`) when you start.

In [ ]:
FILES = {
    "README.md": """\
        # {name}
        > One sentence: who it's for and what it does.

        **Live demo:** <url> · **Write-up:** WRITEUP.md

        ## Architecture
        ```
        user -> UI -> API (FastAPI) -> [retrieval / tools] -> LLM -> guardrails -> answer
                                   \\-> tracing (Phoenix/Langfuse), cost & latency logs
        ```
        ## Results (test set, n=?)
        | version | score | p95 latency | cost / 1k requests |
        |---|---|---|---|
        | baseline | | | |
        | current | | | |

        ## Run locally
        ```bash
        uv sync && cp .env.example .env
        uv run python evals/run.py          # evals
        uv run uvicorn app.api:app          # API (add when you build it)
        ```
        ## Tracing · Guardrails · Deployment
        Describe what's on and where to see it.
        """,
    "SPEC.md": """\
        # Spec
        - **User & job**: who, what task, how often, what they do today
        - **Input → output**: formats, languages, sizes
        - **Definition of good**: the metric(s) and the bar to ship (e.g. ≥ 90% field accuracy, p95 < 3 s, < $1 / 1k requests)
        - **Constraints**: privacy/PII, cost, latency, offline?, licences
        - **Out of scope**: what this will NOT do
        - **Risks**: what could make this fail, and the cheapest test for each
        """,
    "DECISIONS.md": """\
        # Decision log
        | date | change | why | eval before → after | keep? |
        |---|---|---|---|---|
        | | baseline: single prompt, <model> | simplest thing | – → ? | ✓ |
        """,
    "WRITEUP.md": """\
        # {name}: what I built and what I learned
        1. **Problem** — who has it, why it matters, why existing tools fall short
        2. **Approach** — architecture diagram; the 2–3 key design decisions and the alternatives you rejected (with numbers)
        3. **Evaluation** — how the eval set was built, metrics, why they reflect the user's goal
        4. **Results** — baseline → final table; cost and latency; ablations (what each component contributes)
        5. **Failure analysis** — the remaining error buckets, with real examples
        6. **Limitations & risks** — where not to trust it
        7. **What's next**
        """,
    "evals/dataset.jsonl": "",
    "evals/run.py": '''\
        """Run the eval set and fail (exit 1) below a threshold. Usage: python evals/run.py [--min-score 0.8] [--split dev]"""
        import argparse, json, sys, time
        from pathlib import Path

        sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
        from app.core import predict


        def score(output, expected):
            """Replace with your metric (exact match, field accuracy, rubric judge...). Here: expected string contained in output."""
            return float(expected.lower() in (output or "").lower())


        if __name__ == "__main__":
            ap = argparse.ArgumentParser(); ap.add_argument("--min-score", type=float, default=0.0); ap.add_argument("--split", default=None)
            args = ap.parse_args()
            rows = [json.loads(l) for l in (Path(__file__).parent / "dataset.jsonl").read_text().splitlines() if l.strip()]
            rows = [r for r in rows if args.split in (None, r.get("split"))]
            results, start = [], time.perf_counter()
            for r in rows:
                out = predict(r["input"])
                results.append({**r, "output": out, "score": score(out, r["expected"])})
            mean = sum(x["score"] for x in results) / max(len(results), 1)
            (Path(__file__).parent / "last_run.jsonl").write_text("\\n".join(json.dumps(x, ensure_ascii=False) for x in results))
            for x in results:
                if x["score"] < 1:
                    print(f"FAIL {x['input'][:60]!r} -> {str(x['output'])[:80]!r} (expected {x['expected']!r})")
            print(f"score {mean:.2f} on {len(results)} examples in {time.perf_counter() - start:.1f}s (threshold {args.min_score})")
            sys.exit(0 if mean >= args.min_score else 1)
        ''',
    "app/__init__.py": "",
    "app/core.py": '''\
        """Your system lives here. Start with the simplest baseline; evals/run.py calls predict()."""
        import os

        from openai import OpenAI

        client = OpenAI(base_url=os.getenv("LLM_BASE_URL", "http://localhost:11434/v1"), api_key=os.getenv("LLM_API_KEY", "ollama"))
        MODEL = os.getenv("LLM_MODEL", "qwen3:1.7b")
        SYSTEM = "Answer in one short sentence."


        def predict(text: str) -> str:
            extra = {"reasoning_effort": "none"} if "localhost" in str(client.base_url) else {}
            r = client.chat.completions.create(model=MODEL, temperature=0, max_tokens=200, **extra,
                                               messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}])
            return r.choices[0].message.content
        ''',
    ".env.example": "LLM_BASE_URL=https://generativelanguage.googleapis.com/v1beta/openai/\nLLM_API_KEY=\nLLM_MODEL=gemini-flash-lite-latest\n",
    ".gitignore": ".env\n__pycache__/\n.venv/\nevals/last_run.jsonl\n",
    ".github/workflows/ci.yml": """\
        name: ci
        on: {pull_request: {}, push: {branches: [main]}}
        jobs:
          evals:
            runs-on: ubuntu-latest
            env:
              LLM_BASE_URL: ${{ vars.LLM_BASE_URL }}
              LLM_MODEL: ${{ vars.LLM_MODEL }}
              LLM_API_KEY: ${{ secrets.LLM_API_KEY }}
            steps:
              - uses: actions/checkout@v4
              - uses: astral-sh/setup-uv@v6
              - run: uv run --with openai python evals/run.py --split dev --min-score 0.8
        """,
}


def scaffold(name, examples=()):
    root = DATA / "capstone" / name
    for rel, content in FILES.items():
        p = root / rel
        p.parent.mkdir(parents=True, exist_ok=True)
        if not p.exists():  # never overwrite your edits
            p.write_text(textwrap.dedent(content).replace("{name}", name))
    if examples and not (root / "evals/dataset.jsonl").read_text().strip():
        (root / "evals/dataset.jsonl").write_text("\n".join(json.dumps(e, ensure_ascii=False) for e in examples) + "\n")
    return root


demo = scaffold("demo-capstone", examples=[
    {"input": "What is the capital of Peru?", "expected": "Lima", "split": "dev"},
    {"input": "What is the capital of Uruguay?", "expected": "Montevideo", "split": "dev"},
    {"input": "How many sides does a hexagon have?", "expected": "6", "split": "dev"},
    {"input": "Which planet is known as the red planet?", "expected": "Mars", "split": "test"},
])
for p in sorted(demo.rglob("*")):
    if p.is_file():
        print(p.relative_to(demo))

The eval runner works out of the box (against local Ollama by default; set `LLM_BASE_URL` / `LLM_MODEL` for any OpenAI-compatible API).
Its exit code is what makes CI block a regression:

In [ ]:
r = subprocess.run([sys.executable, "evals/run.py", "--split", "dev", "--min-score", "0.8"], cwd=demo, capture_output=True, text=True)
print(r.stdout, r.stderr[-500:], f"exit code {r.returncode} -> CI {'passes' if r.returncode == 0 else 'fails'}")

Read the failure before touching the model: "A hexagon has six sides." is **correct** — the metric is wrong (it wanted the digit "6").
This happens constantly: a large share of early "model errors" are metric or label bugs. The eval set and scorer are code too —
normalise answers (numbers, case, accents: 14.02's normaliser), accept alternatives (14.05), or use a rubric judge (10.03).

## 4. Readiness check
A quick audit of the M7 deliverables. Run it on your capstone repo every week; it reports what's still missing.

In [ ]:
def readiness(root):
    root = Path(root)
    readme = (root / "README.md").read_text() if (root / "README.md").exists() else ""
    data = [l for l in (root / "evals/dataset.jsonl").read_text().splitlines() if l.strip()] if (root / "evals/dataset.jsonl").exists() else []
    decisions = (root / "DECISIONS.md").read_text().count("\n|") - 2 if (root / "DECISIONS.md").exists() else 0
    code = " ".join(p.read_text(errors="ignore") for p in root.rglob("*.py") if ".venv" not in p.parts)
    checks = {
        "spec written": (root / "SPEC.md").exists() and (root / "SPEC.md").read_text() != textwrap.dedent(FILES["SPEC.md"]),
        "≥ 50 eval examples": len(data) >= 50,
        "dev/test split": {json.loads(l).get("split") for l in data} >= {"dev", "test"},
        "evals in CI with a threshold": "--min-score" in ((root / ".github/workflows/ci.yml").read_text() if (root / ".github/workflows/ci.yml").exists() else ""),
        "≥ 5 logged decisions": decisions >= 5,
        "tracing in code": any(k in code for k in ("phoenix", "langfuse", "opentelemetry", "openinference", "logfire")),
        "guardrails in code": any(k in code.lower() for k in ("presidio", "guard", "moderation", "injection")),
        "container / deploy config": any((root / f).exists() for f in ("Dockerfile", "modal_app.py", "fly.toml", "render.yaml")),
        "public URL in README": "http" in readme.split("Live demo:**")[-1][:200] if "Live demo:**" in readme else False,
        "results table filled": "| baseline |" in readme and "| baseline | |" not in readme,
        "write-up drafted": (root / "WRITEUP.md").exists() and len((root / "WRITEUP.md").read_text()) > 3000,
    }
    for k, ok in checks.items():
        print(f"{'✅' if ok else '⬜'} {k}")
    print(f"\n{sum(checks.values())}/{len(checks)} ready")
    return checks


_ = readiness(demo)

## 5. Error analysis: the loop that actually improves the product
After each eval run, read the failures (`evals/last_run.jsonl`) and tag each with a short cause. Then count. Fix the biggest bucket first;
re-run; log the delta in `DECISIONS.md`. Typical buckets: retrieval missed the document, wrong tool/arguments, format violation, hallucinated
detail, ambiguous question, bad label in the eval set (fix the data!).
```python
from collections import Counter
fails = [json.loads(l) for l in open("evals/last_run.jsonl") if json.loads(l)["score"] < 1]
for f in fails: print(f["input"], "->", f["output"])       # read every one
tags = {...}                                               # you write these: {input: "retrieval miss" | "format" | ...}
print(Counter(tags.values()).most_common())
```

## 6. The write-up
Your write-up is the portfolio piece: hiring managers and collaborators read it more than they run your code. Show **judgement**: the
alternatives you tried, the numbers that decided, what still fails. A results table with a baseline and ablations beats any adjective.
Publish it (blog, GitHub README, HF Space card, LinkedIn post linking to it).

## Milestone M7 — deliverable
- [ ] A multimodal or voice feature (14.x) **or** an advanced deep-dive (15.x) in your capstone
- [ ] Capstone deployed with evals in CI, tracing, guardrails, documented (readiness check all ✅)
- [ ] Public write-up with design decisions and eval results

## Exercises
1. Pick a problem you actually have. Spec → eval set → baseline → iterate → deploy → write-up. Run `scaffold("<your-project>")` to start.
2. Before writing any system code, build the eval set (≥ 50 examples) and score a one-prompt baseline. Record it as the first row of `DECISIONS.md`.
3. Do three error-analysis cycles; each must change the score by a measured amount (up or down) and be logged.
4. Ask someone else to use your deployed app for 15 minutes without help; turn every confusion into an eval example or a UX fix.

In [ ]:
# your code here